<a href="https://colab.research.google.com/github/potato-yen/114-2-Programming-Language/blob/main/HW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW3｜Course Desk：課程公告轉待辦＋番茄鐘

這版把老師範例重整成單頁 Gradio 工作台：

1. 可以貼 Moodle 公告網址，也可以先抓取 Moodle 課程清單，再從課程選公告。
2. 公告內容會先清理，只保留標題、正文與相關連結。
3. Gemini 只輸出結構化任務，使用者可勾選要寫入的項目。
4. 可手動新增待辦。
5. 本週任務可勾選後標記完成。
6. 番茄鐘支援開始、暫停、結束，含工作／休息與輪次紀錄。

Google Sheet URL 已保留為使用者替換後的網址。

In [ ]:
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
               gradio pandas beautifulsoup4 google-generativeai python-dateutil icalendar requests lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 8.8 MB/s eta 0:00:00


In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import json
import uuid
import html
import hashlib
import math
from datetime import datetime as dt, timedelta, date
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil.tz import gettz

import gradio as gr
import google.generativeai as genai
from google.colab import auth, userdata
from google.auth import default
import gspread
from icalendar import Calendar


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Google Auth
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Gemini API Key
api_key = userdata.get("gemini")
if api_key:
    genai.configure(api_key=api_key)

# 使用者已替換好的 Google Sheet URL
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing"

TIMEZONE = "Asia/Taipei"
DEFAULT_MOODLE_BASE = "https://moodle3.ntnu.edu.tw"

# Optional Colab Secrets:
# MOODLE_COOKIE：完整 Cookie 字串，例如 "MoodleSession=xxxx; other=value"
# MOODLE_SESSION：只放 MoodleSession 的 value，程式會自動組成 Cookie
# MOODLE_ICS_URL：Moodle calendar export 的 .ics 網址
MOODLE_COOKIE_SECRET = "MOODLE_COOKIE"
MOODLE_SESSION_SECRET = "MOODLE_SESSION"
MOODLE_ICS_SECRET = "MOODLE_ICS_URL"

sh = gc.open_by_url(SHEET_URL)


In [ ]:
# =========================
# Google Sheet schema
# =========================

TASKS_SHEET = "course_tasks"
LOGS_SHEET = "course_pomodoro_logs"
IMPORTS_SHEET = "announcement_imports"

TASKS_HEADER = [
    "task_id", "title", "course", "due_at", "priority", "status", "est_min",
    "source_url", "source_title", "source_type", "source_hash",
    "notes", "created_at", "updated_at"
]

LOGS_HEADER = [
    "log_id", "task_id", "task_title", "phase", "pomodoro_no",
    "start_at", "end_at", "duration_min", "note", "created_at"
]

IMPORTS_HEADER = [
    "import_id", "url", "page_title", "fetched_at",
    "text_hash", "detected_count", "created_task_ids", "status"
]


def tznow():
    return dt.now(gettz(TIMEZONE))


def now_iso():
    return tznow().isoformat(timespec="seconds")


def ensure_worksheet(spreadsheet, title, header):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=1000, cols=max(len(header) + 5, 20))
        ws.update("A1", [header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update("A1", [header])
        return ws

    current_header = values[0]
    merged = current_header[:]
    for col in header:
        if col not in merged:
            merged.append(col)
    if merged != current_header:
        ws.update("A1", [merged])
    return ws


ws_tasks = ensure_worksheet(sh, TASKS_SHEET, TASKS_HEADER)
ws_logs = ensure_worksheet(sh, LOGS_SHEET, LOGS_HEADER)
ws_imports = ensure_worksheet(sh, IMPORTS_SHEET, IMPORTS_HEADER)


def read_sheet(ws, header):
    rows = ws.get_all_values()
    if not rows:
        return pd.DataFrame(columns=header)

    cols = rows[0]
    data = rows[1:]
    df = pd.DataFrame(data, columns=cols) if data else pd.DataFrame(columns=cols)

    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def append_dicts(ws, header, rows):
    if not rows:
        return
    values = [[str(row.get(col, "")) for col in header] for row in rows]
    ws.append_rows(values, value_input_option="USER_ENTERED")


def update_task_status(task_ids, status="done"):
    task_ids = set(task_ids or [])
    if not task_ids:
        return 0

    df = read_sheet(ws_tasks, TASKS_HEADER)
    if df.empty:
        return 0

    rows = ws_tasks.get_all_values()
    header = rows[0]
    status_col = header.index("status") + 1
    updated_col = header.index("updated_at") + 1

    count = 0
    for idx, row in df.iterrows():
        if str(row.get("task_id")) in task_ids:
            sheet_row = idx + 2
            ws_tasks.update_cell(sheet_row, status_col, status)
            ws_tasks.update_cell(sheet_row, updated_col, now_iso())
            count += 1
    return count


def make_hash(*parts):
    raw = "|".join([str(p or "").strip() for p in parts])
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]


def normalize_priority(priority):
    p = str(priority or "").lower().strip()
    if p in ["high", "h", "高", "urgent", "緊急"]:
        return "high"
    if p in ["low", "l", "低"]:
        return "low"
    return "medium"


def priority_order(priority):
    return {"high": 0, "medium": 1, "low": 2}.get(str(priority).lower(), 1)


def clean_due_at(value):
    value = str(value or "").strip()
    if not value or value.lower() in ["none", "null", "無", "無截止"]:
        return ""

    m = re.search(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})(?:\s+(\d{1,2}):(\d{2}))?", value)
    if m:
        y, mo, d, hh, mm = m.groups()
        if hh and mm:
            return f"{int(y):04d}-{int(mo):02d}-{int(d):02d} {int(hh):02d}:{int(mm):02d}"
        return f"{int(y):04d}-{int(mo):02d}-{int(d):02d}"

    this_year = tznow().year
    m = re.search(r"(\d{1,2})\s*[月/]\s*(\d{1,2})\s*(?:日)?(?:\s*(\d{1,2}):(\d{2}))?", value)
    if m:
        mo, d, hh, mm = m.groups()
        if hh and mm:
            return f"{this_year:04d}-{int(mo):02d}-{int(d):02d} {int(hh):02d}:{int(mm):02d}"
        return f"{this_year:04d}-{int(mo):02d}-{int(d):02d}"

    return value


def task_choice_pairs(tasks_df=None, include_done=False):
    if tasks_df is None:
        tasks_df = read_sheet(ws_tasks, TASKS_HEADER)
    if tasks_df.empty:
        return []

    df = tasks_df.copy()
    if not include_done:
        df = df[df["status"].astype(str).str.lower() != "done"]
    if df.empty:
        return []

    df["_p"] = df["priority"].map(priority_order)
    df["_due"] = df["due_at"].replace("", "9999-12-31")
    df = df.sort_values(["_p", "_due", "created_at"], ascending=[True, True, False])

    pairs = []
    for _, row in df.iterrows():
        due = row.get("due_at") or "無截止"
        title = row.get("title") or "未命名任務"
        course = row.get("course") or "未填課程"
        label = f"{title}｜{course}｜{due}｜{row.get('priority','medium')}"
        pairs.append((label, row.get("task_id")))
    return pairs


def add_task_row(title, course="", due_at="", priority="medium", est_min="25", source_url="", source_title="", source_type="manual", notes=""):
    title = str(title or "").strip()
    if not title:
        raise ValueError("任務名稱不能空白。")

    due_at = clean_due_at(due_at)
    priority = normalize_priority(priority)
    try:
        est_min = int(float(est_min or 25))
    except Exception:
        est_min = 25

    source_hash = make_hash(source_url, title, due_at, source_title)
    existing = read_sheet(ws_tasks, TASKS_HEADER)
    if not existing.empty and source_hash in set(existing["source_hash"].astype(str)) and source_type != "manual":
        return None

    row = {
        "task_id": str(uuid.uuid4())[:8],
        "title": title,
        "course": str(course or "").strip(),
        "due_at": due_at,
        "priority": priority,
        "status": "todo",
        "est_min": str(est_min),
        "source_url": str(source_url or ""),
        "source_title": str(source_title or ""),
        "source_type": source_type,
        "source_hash": source_hash,
        "notes": str(notes or ""),
        "created_at": now_iso(),
        "updated_at": now_iso(),
    }
    append_dicts(ws_tasks, TASKS_HEADER, [row])
    return row


def append_pomodoro_log(task_id, task_title, phase, pomodoro_no, start_at, end_at, duration_min, note=""):
    row = {
        "log_id": str(uuid.uuid4())[:8],
        "task_id": task_id or "",
        "task_title": task_title or "",
        "phase": phase or "work",
        "pomodoro_no": str(pomodoro_no or 0),
        "start_at": start_at or "",
        "end_at": end_at or now_iso(),
        "duration_min": str(duration_min or 0),
        "note": note or "",
        "created_at": now_iso(),
    }
    append_dicts(ws_logs, LOGS_HEADER, [row])
    return row


In [ ]:
# =========================
# Moodle crawling and parsing
# =========================


def get_moodle_headers():
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept-Language": "zh-TW,zh;q=0.9,en;q=0.8",
    }

    cookie = ""
    try:
        cookie = userdata.get(MOODLE_COOKIE_SECRET) or ""
    except Exception:
        cookie = ""

    if not cookie:
        try:
            session_value = userdata.get(MOODLE_SESSION_SECRET) or ""
            if session_value:
                cookie = f"MoodleSession={session_value}"
        except Exception:
            cookie = ""

    if cookie:
        headers["Cookie"] = cookie
    return headers


def moodle_get(url, timeout=25):
    resp = requests.get(url, headers=get_moodle_headers(), timeout=timeout, allow_redirects=True)
    resp.raise_for_status()
    return resp


def compact_text(text, max_chars=6500):
    text = re.sub(r"\s+", " ", text or "").strip()
    noise_patterns = [
        r"跳到主要內容", r"You are logged in as[^。]*", r"上一頁", r"下一頁",
        r"Moodle Docs[^。]*", r"正體中文", r"English"
    ]
    for pat in noise_patterns:
        text = re.sub(pat, " ", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]


def extract_page_title(soup):
    for selector in [".page-header-headings h1", "#page-header h1", "h1", "[role='main'] h2", "title"]:
        node = soup.select_one(selector)
        if node:
            t = compact_text(node.get_text(" ", strip=True), 180)
            if t:
                return t
    return "未取得標題"


def remove_moodle_noise(node):
    for selector in [
        "script", "style", "noscript", "svg", "canvas", "nav", "footer",
        ".navbar", ".breadcrumb", ".drawer", ".secondary-navigation",
        ".activity-navigation", ".urlselect", ".skiplinks", ".sr-only",
        ".commands", ".post-actions", ".ratingform", ".forum-post-rating"
    ]:
        for tag in node.select(selector):
            tag.decompose()


def extract_links(node, base_url, limit=30):
    links = []
    seen = set()
    for a in node.select("a[href]"):
        href = a.get("href", "").strip()
        label = compact_text(a.get_text(" ", strip=True), 120)
        if not href:
            continue
        full = urljoin(base_url, href)
        if not label:
            label = full
        key = (label, full)
        if key in seen:
            continue
        seen.add(key)
        links.append({"text": label, "href": full})
        if len(links) >= limit:
            break
    return links


def parse_course_list(base_url=DEFAULT_MOODLE_BASE):
    base_url = (base_url or DEFAULT_MOODLE_BASE).strip().rstrip("/")
    urls = [f"{base_url}/my/courses.php", f"{base_url}/my/", f"{base_url}/"]
    found = []
    seen = set()

    for url in urls:
        try:
            resp = moodle_get(url)
        except Exception:
            continue
        soup = BeautifulSoup(resp.text, "html.parser")
        remove_moodle_noise(soup)
        for a in soup.select("a[href*='course/view.php']"):
            href = urljoin(url, a.get("href", ""))
            title = compact_text(a.get_text(" ", strip=True), 160)
            m = re.search(r"[?&]id=(\d+)", href)
            cid = m.group(1) if m else href
            if not title or cid in seen:
                continue
            if len(title) < 3:
                continue
            seen.add(cid)
            found.append({"id": cid, "title": title, "url": href})
        if found:
            break
    return found


def parse_forum_list(url, html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    remove_moodle_noise(soup)
    page_title = extract_page_title(soup)

    announcements = []
    seen = set()

    for a in soup.select("a[href*='mod/forum/discuss.php']"):
        href = urljoin(url, a.get("href", ""))
        title = compact_text(a.get_text(" ", strip=True), 160)
        if not title or href in seen:
            continue
        seen.add(href)
        parent_text = compact_text(a.find_parent("tr").get_text(" ", strip=True), 260) if a.find_parent("tr") else ""
        announcements.append({
            "title": title,
            "url": href,
            "meta": parent_text,
        })

    return page_title, announcements


def parse_course_announcements(course_url):
    resp = moodle_get(course_url)
    soup = BeautifulSoup(resp.text, "html.parser")
    remove_moodle_noise(soup)
    page_title = extract_page_title(soup)

    forums = []
    seen = set()
    for a in soup.select("a[href*='mod/forum/view.php']"):
        href = urljoin(course_url, a.get("href", ""))
        title = compact_text(a.get_text(" ", strip=True), 160)
        if not href or href in seen:
            continue
        # 優先保留公告、消息、News forum；但也保留一般 forum 作為 fallback
        seen.add(href)
        score = 0
        if re.search(r"公告|消息|news|announcement|announcements", title, flags=re.I):
            score += 2
        forums.append({"title": title or href, "url": href, "score": score})

    forums = sorted(forums, key=lambda x: x.get("score", 0), reverse=True)
    return page_title, forums


def extract_discussion_content(url, html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    remove_moodle_noise(soup)
    page_title = extract_page_title(soup)

    # Moodle forum discussion usually stores the first post in .forumpost .post-content or .posting
    candidates = []
    for selector in [
        ".forumpost .post-content", ".forumpost .posting", ".forum-post-core",
        "[data-region='post-content']", ".discussion-content", "[role='main']", "#region-main", "main", "body"
    ]:
        for node in soup.select(selector):
            text = compact_text(node.get_text("\n", strip=True), 6500)
            if len(text) > 80:
                candidates.append((len(text), node, text))
        if candidates:
            break

    if candidates:
        _, main, text = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
    else:
        main = soup.body or soup
        text = compact_text(main.get_text("\n", strip=True), 6500)

    links = extract_links(main, url, 25)
    return page_title, text, links


def fetch_url_as_announcement(url):
    url = (url or "").strip()
    if not url:
        raise ValueError("請輸入網址。")
    resp = moodle_get(url)
    final_url = resp.url
    html_text = resp.text

    # 若是 forum view，一般是公告列表，不直接把整頁拿去餵 AI。
    if "mod/forum/view.php" in final_url:
        page_title, items = parse_forum_list(final_url, html_text)
        if items:
            return {
                "kind": "forum_list",
                "url": final_url,
                "title": page_title,
                "items": items,
                "text": "",
                "links": [],
            }

    page_title, text, links = extract_discussion_content(final_url, html_text)
    return {
        "kind": "announcement",
        "url": final_url,
        "title": page_title,
        "text": text,
        "links": links,
    }


In [ ]:
# =========================
# Gemini extraction
# =========================


def generate_with_gemini(prompt):
    if not api_key:
        raise RuntimeError("尚未設定 Gemini API Key。請在 Colab Secrets 設定 gemini。")
    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)
    return response.text or ""


def parse_json_safely(text):
    text = (text or "").strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if m:
            return json.loads(m.group(0))
        raise


def extract_tasks_with_gemini(announcement):
    if not announcement or not announcement.get("text"):
        raise ValueError("尚未載入可解析的公告內容。")

    links_text = "\n".join([f"- {x.get('text','')}: {x.get('href','')}" for x in announcement.get("links", [])[:15]])
    prompt = f"""
你是一個課程公告待辦事項抽取器。請從 Moodle 課程公告中判斷是否包含作業、考試、報告、繳交、分組、閱讀、截止日期或需要學生處理的事項。

請只輸出 JSON，不要輸出 Markdown，不要解釋。

JSON 格式：
{{
  "has_task": true,
  "course": "課程名稱，若不確定可從標題推測",
  "tasks": [
    {{
      "title": "任務名稱，12字到24字，清楚可執行",
      "course": "課程名稱",
      "due_at": "YYYY-MM-DD HH:MM；若沒有截止日則空字串",
      "priority": "high / medium / low",
      "est_min": 25,
      "note": "備註，60字內，包含依據或注意事項",
      "source_evidence": "公告原文中的短依據，40字內"
    }}
  ]
}}

規則：
1. 不要把作者、回應數、瀏覽數、導覽列文字當成任務。
2. 若公告只是一般資訊，has_task = false 並回傳空 tasks。
3. 若有多個待辦，拆成多筆任務。
4. 截止日不確定就留空，不要捏造日期。
5. 優先順序：明確考試/截止/繳交為 high；一般分組或準備為 medium；參考閱讀為 low。

公告標題：{announcement.get('title','')}
公告網址：{announcement.get('url','')}
相關連結：
{links_text}

公告正文：
{announcement.get('text','')[:5500]}
""".strip()

    raw = generate_with_gemini(prompt)
    data = parse_json_safely(raw)
    tasks = data.get("tasks", []) or []

    cleaned = []
    default_course = data.get("course") or announcement.get("title", "")
    for t in tasks:
        title = str(t.get("title", "")).strip()
        if not title:
            continue
        course = str(t.get("course") or default_course or "").strip()
        due_at = clean_due_at(t.get("due_at", ""))
        priority = normalize_priority(t.get("priority", "medium"))
        try:
            est_min = int(float(t.get("est_min", 25) or 25))
        except Exception:
            est_min = 25
        note = str(t.get("note") or "").strip()
        evidence = str(t.get("source_evidence") or "").strip()
        if evidence and evidence not in note:
            note = f"{note}｜依據：{evidence}" if note else f"依據：{evidence}"
        cleaned.append({
            "title": title,
            "course": course,
            "due_at": due_at,
            "priority": priority,
            "est_min": est_min,
            "note": note[:160],
            "source_evidence": evidence,
            "source_url": announcement.get("url", ""),
            "source_title": announcement.get("title", ""),
        })
    return cleaned


In [ ]:
# =========================
# Moodle calendar ICS sync
# =========================


def sync_moodle_ics(ics_url):
    ics_url = (ics_url or "").strip()
    if not ics_url:
        try:
            ics_url = userdata.get(MOODLE_ICS_SECRET) or ""
        except Exception:
            ics_url = ""

    if not ics_url:
        raise ValueError("請輸入 Moodle Calendar ICS URL，或在 Colab Secrets 設定 MOODLE_ICS_URL。")

    resp = requests.get(ics_url, timeout=25)
    resp.raise_for_status()
    cal = Calendar.from_ical(resp.content)

    rows = []
    existing = read_sheet(ws_tasks, TASKS_HEADER)
    existing_hashes = set(existing["source_hash"].astype(str)) if not existing.empty else set()

    for comp in cal.walk():
        if comp.name != "VEVENT":
            continue
        summary = str(comp.get("summary", "")).strip()
        if not summary:
            continue

        dtstart = comp.get("dtstart")
        dtend = comp.get("dtend")
        event_dt = dtend.dt if dtend else (dtstart.dt if dtstart else None)
        if isinstance(event_dt, dt):
            due_at = event_dt.astimezone(gettz(TIMEZONE)).strftime("%Y-%m-%d %H:%M")
        elif isinstance(event_dt, date):
            due_at = event_dt.strftime("%Y-%m-%d")
        else:
            due_at = ""

        # 只抓未來與近期事件，避免整個歷史資料灌進去
        if due_at:
            try:
                due_date = pd.to_datetime(due_at).date()
                if due_date < (tznow().date() - timedelta(days=2)):
                    continue
            except Exception:
                pass

        source_hash = make_hash("ics", summary, due_at)
        if source_hash in existing_hashes:
            continue

        rows.append({
            "task_id": str(uuid.uuid4())[:8],
            "title": summary[:80],
            "course": "",
            "due_at": due_at,
            "priority": "high" if due_at else "medium",
            "status": "todo",
            "est_min": "25",
            "source_url": ics_url,
            "source_title": "Moodle Calendar",
            "source_type": "moodle_calendar",
            "source_hash": source_hash,
            "notes": "由 Moodle 行事曆匯入。",
            "created_at": now_iso(),
            "updated_at": now_iso(),
        })
        existing_hashes.add(source_hash)

    append_dicts(ws_tasks, TASKS_HEADER, rows)
    return rows


In [ ]:
# =========================
# HTML render helpers
# =========================


def esc(x):
    return html.escape(str(x if x is not None else ""))


def status_html(message, kind=""):
    cls = "status" + (f" {kind}" if kind else "")
    return f"<div class='{cls}'>{esc(message)}</div>"


def announcement_preview_html(announcement):
    if not announcement:
        return "<div class='empty-note'>尚未載入公告。</div>"
    if announcement.get("kind") == "forum_list":
        items = announcement.get("items", [])
        lis = "".join([f"<li><a href='{esc(x['url'])}' target='_blank'>{esc(x['title'])}</a><div class='mini'>{esc(x.get('meta',''))}</div></li>" for x in items[:12]])
        return f"""
        <div class='announcement-card'>
          <div class='small-kicker'>公告列表</div>
          <h3>{esc(announcement.get('title'))}</h3>
          <div class='source-url'>{esc(announcement.get('url'))}</div>
          <p class='soft-text'>這是一個公告列表頁。請在下方選擇單則公告後載入內容，再交給 Gemini 解析。</p>
          <ul class='link-list'>{lis}</ul>
        </div>
        """
    links = announcement.get("links", []) or []
    link_html = "".join([f"<li><a href='{esc(x.get('href'))}' target='_blank'>{esc(x.get('text'))}</a></li>" for x in links[:8]])
    text = esc(announcement.get("text", ""))
    return f"""
    <div class='announcement-card'>
      <div class='small-kicker'>公告標題</div>
      <h3>{esc(announcement.get('title',''))}</h3>
      <div class='source-url'>{esc(announcement.get('url',''))}</div>
      <p>{text[:1200]}{'…' if len(text) > 1200 else ''}</p>
      <div class='small-kicker'>相關連結</div>
      <ul class='link-list'>{link_html or '<li class="soft-text">沒有擷取到明確連結</li>'}</ul>
    </div>
    """


def detected_tasks_table_html(tasks):
    tasks = tasks or []
    if not tasks:
        return "<div class='empty-note'>尚未解析任務。</div>"
    rows = []
    for i, t in enumerate(tasks, start=1):
        rows.append(f"""
        <tr>
          <td>{i}</td>
          <td>{esc(t.get('title'))}</td>
          <td>{esc(t.get('course'))}</td>
          <td>{esc(t.get('due_at') or '無截止')}</td>
          <td>{esc(t.get('priority'))}</td>
          <td>{esc(t.get('est_min'))}</td>
          <td>{esc(t.get('note'))}</td>
        </tr>
        """)
    return f"""
    <div class='table-wrap'>
      <table class='soft-table'>
        <thead><tr><th>序號</th><th>任務</th><th>課程</th><th>截止</th><th>優先</th><th>估時</th><th>備註</th></tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """


def tasks_preview_table_html():
    df = read_sheet(ws_tasks, TASKS_HEADER)
    if df.empty:
        return "<div class='empty-note'>目前沒有任務。</div>"
    df = df[df["status"].astype(str).str.lower() != "done"].copy()
    if df.empty:
        return "<div class='empty-note'>目前沒有未完成任務。</div>"
    df["_p"] = df["priority"].map(priority_order)
    df["_due"] = df["due_at"].replace("", "9999-12-31")
    df = df.sort_values(["_p", "_due", "created_at"], ascending=[True, True, False]).head(20)
    rows = []
    for _, r in df.iterrows():
        rows.append(f"""
        <tr>
          <td>{esc(r.get('title'))}</td>
          <td>{esc(r.get('course'))}</td>
          <td>{esc(r.get('due_at') or '無截止')}</td>
          <td>{esc(r.get('priority'))}</td>
          <td>{esc(r.get('status'))}</td>
          <td>{esc(r.get('source_type'))}</td>
        </tr>
        """)
    return f"""
    <div class='table-wrap'>
      <table class='soft-table'>
        <thead><tr><th>任務</th><th>課程</th><th>截止</th><th>優先</th><th>狀態</th><th>來源</th></tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """


def timer_display_html(remaining=25*60, phase="work", count=0, caption="選擇任務後開始一輪專注。"):
    remaining = int(remaining or 0)
    m, s = divmod(max(remaining, 0), 60)
    phase_text = "專注" if phase == "work" else "休息"
    return f"""
    <div class='timer-face'>
      <div class='timer-mode' id='cd-timer-mode'>{esc(phase_text)} · 第 {int(count or 0) + 1} 輪</div>
      <div class='timer-display' id='cd-timer-display'>{m:02d}:{s:02d}</div>
      <div class='timer-caption' id='cd-timer-caption'>{esc(caption)}</div>
    </div>
    """


In [ ]:
# =========================
# UI event functions
# =========================


def ui_fetch_courses(base_url):
    try:
        courses = parse_course_list(base_url or DEFAULT_MOODLE_BASE)
        choices = [(f"{c['title']}｜id={c['id']}", c["url"]) for c in courses]
        if not choices:
            return courses, gr.update(choices=[], value=None), status_html("沒有抓到課程。請確認 MoodleSession 是否有效。", "error")
        return courses, gr.update(choices=choices, value=choices[0][1]), status_html(f"已抓到 {len(courses)} 門課程。", "ok")
    except Exception as e:
        return [], gr.update(choices=[], value=None), status_html(f"抓取課程失敗：{e}", "error")


def ui_fetch_course_forums(course_url):
    if not course_url:
        return [], gr.update(choices=[], value=None), status_html("請先選擇課程。", "error")
    try:
        page_title, forums = parse_course_announcements(course_url)
        if not forums:
            return [], gr.update(choices=[], value=None), status_html("這門課沒有抓到 forum/公告連結。", "error")
        choices = [(f"{x['title']}", x["url"]) for x in forums]
        return forums, gr.update(choices=choices, value=choices[0][1]), status_html(f"已從「{page_title}」抓到 {len(forums)} 個公告/討論區連結。", "ok")
    except Exception as e:
        return [], gr.update(choices=[], value=None), status_html(f"抓取課程公告入口失敗：{e}", "error")


def ui_fetch_announcement_url(url):
    try:
        ann = fetch_url_as_announcement(url)
        if ann.get("kind") == "forum_list":
            items = ann.get("items", [])
            choices = [(f"{x['title']}", x["url"]) for x in items]
            return ann, items, gr.update(choices=choices, value=choices[0][1] if choices else None), announcement_preview_html(ann), status_html(f"已抓到公告列表，共 {len(items)} 則。請選擇單則公告載入。", "ok")
        return ann, [], gr.update(choices=[], value=None), announcement_preview_html(ann), status_html("已載入單則公告，可以解析任務。", "ok")
    except Exception as e:
        return None, [], gr.update(choices=[], value=None), announcement_preview_html(None), status_html(f"抓取公告失敗：{e}", "error")


def ui_load_selected_announcement(announcement_url):
    if not announcement_url:
        return None, announcement_preview_html(None), status_html("請先選擇一則公告。", "error")
    try:
        ann = fetch_url_as_announcement(announcement_url)
        if ann.get("kind") == "forum_list":
            return None, announcement_preview_html(ann), status_html("你選到的仍是列表頁，請再選單則 discuss.php 公告。", "error")
        return ann, announcement_preview_html(ann), status_html("已載入選取公告，可以解析任務。", "ok")
    except Exception as e:
        return None, announcement_preview_html(None), status_html(f"載入公告失敗：{e}", "error")


def ui_extract_tasks(announcement):
    try:
        tasks = extract_tasks_with_gemini(announcement)
        choices = [(f"{i+1}. {t['title']}｜{t.get('due_at') or '無截止'}", str(i)) for i, t in enumerate(tasks)]
        if not tasks:
            return [], detected_tasks_table_html([]), gr.update(choices=[], value=[]), status_html("Gemini 沒有判斷出明確待辦。", "ok")
        return tasks, detected_tasks_table_html(tasks), gr.update(choices=choices, value=[c[1] for c in choices]), status_html(f"已解析出 {len(tasks)} 筆可能任務。請勾選要寫入的項目。", "ok")
    except Exception as e:
        return [], detected_tasks_table_html([]), gr.update(choices=[], value=[]), status_html(f"解析失敗：{e}", "error")


def ui_save_selected_tasks(selected_indices, detected_tasks, announcement):
    selected_indices = selected_indices or []
    detected_tasks = detected_tasks or []
    if not selected_indices:
        return tasks_preview_table_html(), gr.update(choices=task_choice_pairs(), value=[]), gr.update(choices=task_choice_pairs(), value=None), status_html("請先勾選要寫入的任務。", "error")

    rows = []
    created_ids = []
    for idx_s in selected_indices:
        try:
            t = detected_tasks[int(idx_s)]
        except Exception:
            continue
        row = add_task_row(
            title=t.get("title"),
            course=t.get("course"),
            due_at=t.get("due_at"),
            priority=t.get("priority"),
            est_min=t.get("est_min"),
            source_url=t.get("source_url") or (announcement or {}).get("url", ""),
            source_title=t.get("source_title") or (announcement or {}).get("title", ""),
            source_type="moodle_announcement",
            notes=t.get("note", ""),
        )
        if row:
            rows.append(row)
            created_ids.append(row["task_id"])

    # 記錄匯入紀錄
    if announcement:
        append_dicts(ws_imports, IMPORTS_HEADER, [{
            "import_id": str(uuid.uuid4())[:8],
            "url": announcement.get("url", ""),
            "page_title": announcement.get("title", ""),
            "fetched_at": now_iso(),
            "text_hash": make_hash(announcement.get("text", "")),
            "detected_count": str(len(detected_tasks)),
            "created_task_ids": ",".join(created_ids),
            "status": "imported",
        }])

    choices = task_choice_pairs()
    msg = f"已寫入 {len(rows)} 筆任務。" if rows else "沒有新增任務；可能是重複公告或已存在。"
    kind = "ok" if rows else "error"
    return tasks_preview_table_html(), gr.update(choices=choices, value=[]), gr.update(choices=choices, value=None), status_html(msg, kind)


def ui_add_manual_task(title, course, due_at, priority, est_min, notes):
    try:
        row = add_task_row(title, course, due_at, priority, est_min, "", "手動新增", "manual", notes)
        choices = task_choice_pairs()
        return "", "", "", "medium", 25, "", tasks_preview_table_html(), gr.update(choices=choices, value=[]), gr.update(choices=choices, value=row["task_id"] if row else None), status_html("已新增手動任務。", "ok")
    except Exception as e:
        return title, course, due_at, priority, est_min, notes, tasks_preview_table_html(), gr.update(choices=task_choice_pairs(), value=[]), gr.update(choices=task_choice_pairs(), value=None), status_html(f"新增失敗：{e}", "error")


def ui_refresh_tasks():
    choices = task_choice_pairs()
    return tasks_preview_table_html(), gr.update(choices=choices, value=[]), gr.update(choices=choices, value=None), status_html("已重新整理任務。", "ok")


def ui_mark_tasks_done(selected_task_ids):
    try:
        count = update_task_status(selected_task_ids or [], "done")
        choices = task_choice_pairs()
        return tasks_preview_table_html(), gr.update(choices=choices, value=[]), gr.update(choices=choices, value=None), status_html(f"已標記 {count} 筆任務完成。", "ok")
    except Exception as e:
        return tasks_preview_table_html(), gr.update(choices=task_choice_pairs(), value=[]), gr.update(choices=task_choice_pairs(), value=None), status_html(f"標記失敗：{e}", "error")


def ui_sync_ics(ics_url):
    try:
        rows = sync_moodle_ics(ics_url)
        choices = task_choice_pairs()
        return tasks_preview_table_html(), gr.update(choices=choices, value=[]), gr.update(choices=choices, value=None), status_html(f"已從行事曆新增 {len(rows)} 筆任務。", "ok")
    except Exception as e:
        return tasks_preview_table_html(), gr.update(choices=task_choice_pairs(), value=[]), gr.update(choices=task_choice_pairs(), value=None), status_html(f"同步失敗：{e}", "error")


def task_title_by_id(task_id):
    df = read_sheet(ws_tasks, TASKS_HEADER)
    if df.empty or not task_id:
        return ""
    row = df[df["task_id"].astype(str) == str(task_id)]
    if row.empty:
        return ""
    return str(row.iloc[0].get("title", ""))


In [ ]:
# =========================
# Timer event functions
# =========================


def _safe_int(value, default):
    try:
        return int(float(value))
    except Exception:
        return default


def start_or_resume_timer(selected_task_id, work_min, break_min, remaining, running, phase, start_at, current_task_id, task_title, count, work_total, break_total):
    work_min = _safe_int(work_min, 25)
    break_min = _safe_int(break_min, 5)
    count = _safe_int(count, 0)

    if not selected_task_id:
        base = work_min * 60
        return (
            base, False, phase or "work", start_at or "", current_task_id or "", task_title or "",
            count, work_min, break_min,
            timer_display_html(base, phase or "work", count, "請先選擇任務。"),
            status_html("請先選擇一個任務。", "error"),
            gr.Timer(active=False)
        )

    try:
        title = task_title_by_id(selected_task_id) or task_title or "未命名任務"
    except Exception:
        title = task_title or "未命名任務"

    # 同一個任務、暫停後繼續：沿用剩餘秒數。
    # 換任務或第一次開始：重新從專注時間開始。
    same_task = str(selected_task_id) == str(current_task_id or "")
    if same_task and remaining and phase in ["work", "break"] and not running:
        new_remaining = int(remaining)
        new_phase = phase
        new_start_at = start_at or now_iso()
    else:
        new_remaining = work_min * 60
        new_phase = "work"
        new_start_at = now_iso()

    return (
        new_remaining, True, new_phase, new_start_at, selected_task_id, title,
        count, work_min, break_min,
        timer_display_html(new_remaining, new_phase, count, "計時中。"),
        status_html("番茄鐘已開始／繼續。", "ok"),
        gr.Timer(active=True)
    )


def pause_timer(remaining, running, phase, count):
    return (
        False,
        timer_display_html(int(remaining or 0), phase or "work", count or 0, "已暫停。按開始可繼續。"),
        status_html("已暫停。", "ok"),
        gr.Timer(active=False)
    )


def finish_current_timer(remaining, running, phase, start_at, task_id, task_title, count, work_total, break_total, note):
    phase = phase or "work"
    work_total = _safe_int(work_total, 25)
    break_total = _safe_int(break_total, 5)
    remaining = int(remaining or 0)
    total_sec = (work_total if phase == "work" else break_total) * 60
    elapsed_sec = max(total_sec - remaining, 0)
    duration_min = max(1, math.ceil(elapsed_sec / 60)) if elapsed_sec > 0 else 0

    if task_id and duration_min > 0:
        append_pomodoro_log(task_id, task_title or task_title_by_id(task_id), phase, int(count or 0) + 1, start_at or now_iso(), now_iso(), duration_min, note or "")
        msg = f"已結束並紀錄 {duration_min} 分鐘。"
    else:
        msg = "已結束，但沒有有效計時可紀錄。"

    reset_sec = work_total * 60
    return (
        reset_sec, False, "work", "", task_id or "", task_title or "", int(count or 0),
        timer_display_html(reset_sec, "work", count or 0, "已結束。可開始下一輪。"),
        status_html(msg, "ok"),
        gr.Timer(active=False)
    )


def timer_tick(remaining, running, phase, start_at, task_id, task_title, count, work_total, break_total, note):
    remaining = int(remaining or 0)
    count = int(count or 0)
    phase = phase or "work"
    work_total = _safe_int(work_total, 25)
    break_total = _safe_int(break_total, 5)

    if not running:
        return (
            remaining, running, phase, start_at, task_id, task_title, count,
            timer_display_html(remaining, phase, count, "已暫停。"),
            gr.Timer(active=False)
        )

    remaining -= 1
    if remaining > 0:
        return (
            remaining, True, phase, start_at, task_id, task_title, count,
            timer_display_html(remaining, phase, count, "計時中。"),
            gr.Timer(active=True)
        )

    # 時間到：專注完成時先寫入紀錄，再自動進入休息。
    if phase == "work":
        if task_id:
            append_pomodoro_log(task_id, task_title or task_title_by_id(task_id), "work", count + 1, start_at or now_iso(), now_iso(), work_total, note or "")
        count += 1
        if break_total > 0:
            return (
                break_total * 60, True, "break", now_iso(), task_id, task_title, count,
                timer_display_html(break_total * 60, "break", count, "專注完成，進入休息。"),
                gr.Timer(active=True)
            )
        return (
            work_total * 60, False, "work", "", task_id, task_title, count,
            timer_display_html(work_total * 60, "work", count, "專注完成。可開始下一輪。"),
            gr.Timer(active=False)
        )

    # 休息結束
    return (
        work_total * 60, False, "work", "", task_id, task_title, count,
        timer_display_html(work_total * 60, "work", count, "休息結束。按開始進入下一輪。"),
        gr.Timer(active=False)
    )

In [ ]:
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+TC:wght@400;500;700&family=Cormorant+Garamond:wght@600;700&display=swap');

:root {
  --paper: #f5f3ee;
  --paper-soft: #fbf8f1;
  --paper-deep: #e8dfcf;
  --mist: #dbe8ef;
  --sage: #dbe8df;
  --ink: #171717;
  --ink-soft: #5e574f;
  --line: #cfc5b5;
  --button: #111111;
}

* { color-scheme: light !important; }
html, body, gradio-app, .gradio-container, #root {
  background: var(--paper) !important;
  color: var(--ink) !important;
  font-family: 'Noto Sans TC', sans-serif !important;
}

.gradio-container {
  max-width: none !important;
}

#desk-shell {
  max-width: 1280px;
  margin: 0 auto;
  padding: 20px 8px 40px;
}

.hero-wrap {
  padding: 18px 4px 28px;
}
.eyebrow {
  font-size: 14px;
  color: var(--ink-soft) !important;
  margin-bottom: 8px;
}
.hero-title {
  font-family: 'Cormorant Garamond', 'Noto Sans TC', serif !important;
  font-size: 70px;
  line-height: .92;
  margin: 0;
  color: var(--ink) !important;
}
.hero-subtitle {
  margin-top: 18px;
  max-width: 780px;
  color: var(--ink-soft) !important;
  line-height: 1.8;
  font-size: 18px;
}

.panel {
  background: var(--paper-soft) !important;
  border: 1px solid var(--line) !important;
  border-radius: 28px !important;
  padding: 18px !important;
  box-shadow: none !important;
  overflow: hidden !important;
}
.panel-warm { background: var(--paper-deep) !important; }
.panel-blue { background: var(--mist) !important; }
.panel-green { background: var(--sage) !important; }

.panel > div,
.panel .block,
.panel .form,
.panel .gr-form,
.panel .gr-box,
.panel .gr-panel,
.panel .wrap,
.panel .contain,
.panel .input-container,
.panel .output-container,
.panel [data-testid="block-root"],
.panel [data-testid="form"] {
  background: transparent !important;
  color: var(--ink) !important;
  border-color: transparent !important;
  box-shadow: none !important;
}

.section-title h3 {
  font-family: 'Cormorant Garamond', 'Noto Sans TC', serif !important;
  font-size: 36px;
  line-height: 1.05;
  margin: 0 0 6px 0;
  color: var(--ink) !important;
}
.section-note {
  color: var(--ink-soft) !important;
  font-size: 14px;
  line-height: 1.7;
  margin-bottom: 14px;
}

.gradio-container,
.gradio-container p,
.gradio-container span,
.gradio-container div,
.gradio-container label,
.gradio-container .markdown,
.gradio-container .prose,
.gradio-container .html-container {
  color: var(--ink) !important;
}

.gradio-container label,
.gradio-container .block-label,
.gradio-container .block-title,
.gradio-container .label-wrap {
  color: var(--ink) !important;
  font-weight: 600 !important;
}

textarea, input, select,
.gr-textbox textarea,
.gr-textbox input,
.gr-number input,
.gr-dropdown input {
  background: var(--paper-soft) !important;
  color: var(--ink) !important;
  border: 1px solid var(--line) !important;
  border-radius: 16px !important;
  box-shadow: none !important;
}
textarea::placeholder,
input::placeholder { color: #887e72 !important; }

/* Dropdown / checkbox / radio inner layers */
.gr-dropdown,
.gr-checkboxgroup,
.gr-radio,
.gr-form,
.gr-input,
.gr-textbox,
.gr-number {
  background: transparent !important;
  color: var(--ink) !important;
}

button.primary-btn {
  background: var(--button) !important;
  color: #fff !important;
  border-radius: 999px !important;
  border: 1px solid var(--button) !important;
  font-weight: 700 !important;
}
button.secondary-btn {
  background: transparent !important;
  color: var(--ink) !important;
  border-radius: 999px !important;
  border: 1px solid var(--ink-soft) !important;
  font-weight: 700 !important;
}
button { box-shadow: none !important; }

.status {
  border: 1px solid var(--line) !important;
  background: rgba(255,255,255,.62) !important;
  border-radius: 18px !important;
  padding: 12px 16px !important;
  color: var(--ink-soft) !important;
  font-size: 14px !important;
}
.status.ok { background: rgba(219,232,223,.82) !important; color: var(--ink) !important; }
.status.error { background: rgba(232,222,222,.82) !important; color: var(--ink) !important; }
.empty-note { padding: 18px 2px; color: var(--ink-soft) !important; }

.announcement-card {
  background: rgba(255,255,255,.55) !important;
  border: 1px solid rgba(23,23,23,.08) !important;
  border-radius: 22px !important;
  padding: 18px !important;
}
.announcement-card h3 {
  font-family: 'Cormorant Garamond', 'Noto Sans TC', serif !important;
  font-size: 30px;
  margin: 2px 0 8px;
  color: var(--ink) !important;
}
.announcement-card p { line-height: 1.75; }
.small-kicker { font-size: 13px; color: var(--ink-soft) !important; margin: 8px 0 5px; }
.source-url, .mini { color: var(--ink-soft) !important; font-size: 12px; overflow-wrap: anywhere; }
.link-list { padding-left: 18px; }
.link-list a { color: var(--ink) !important; text-decoration: underline; text-underline-offset: 3px; }

.table-wrap {
  overflow-x: auto;
  border: 1px solid var(--line);
  border-radius: 20px;
  background: rgba(255,255,255,.42);
}
.soft-table {
  width: 100%;
  border-collapse: collapse;
  color: var(--ink) !important;
  font-size: 14px;
}
.soft-table th,
.soft-table td {
  border: 1px solid var(--line);
  padding: 12px 14px;
  vertical-align: top;
  background: rgba(255,255,255,.18) !important;
  color: var(--ink) !important;
}
.soft-table th {
  background: rgba(255,255,255,.62) !important;
  font-weight: 700;
}

.timer-face {
  background: rgba(255,255,255,.44) !important;
  border: 1px solid var(--line) !important;
  border-radius: 30px !important;
  padding: 34px 20px !important;
  text-align: center !important;
  margin-bottom: 18px;
}
.timer-mode { color: var(--ink-soft) !important; font-size: 14px; margin-bottom: 8px; }
.timer-display {
  font-family: 'Cormorant Garamond', 'Noto Sans TC', serif !important;
  font-size: 82px;
  line-height: 1;
  color: var(--ink) !important;
}
.timer-caption { color: var(--ink-soft) !important; margin-top: 10px; font-size: 14px; }


/* Multiselect dropdown: replace unstable CheckboxGroup selection */
.gr-dropdown,
.gr-dropdown > div,
.gr-dropdown .wrap,
.gr-dropdown .container,
.gr-dropdown .input-container {
  background: var(--paper-soft) !important;
  color: var(--ink) !important;
  border-color: var(--line) !important;
}

.gr-dropdown label,
.gr-dropdown span,
.gr-dropdown div {
  color: var(--ink) !important;
}

.gr-dropdown [data-testid="token"],
.gr-dropdown .token,
.gr-dropdown .item {
  background: rgba(232,223,207,.75) !important;
  color: var(--ink) !important;
  border: 1px solid var(--line) !important;
  border-radius: 999px !important;
}



/* Gradio toast / error notification: keep it readable if backend throws. */
.toast-wrap, .toast, [data-testid="toast"], [data-testid="toast-container"] {
  background: var(--paper-soft) !important;
  color: var(--ink) !important;
  border: 1px solid var(--line) !important;
  border-radius: 14px !important;
}
.toast button, [data-testid="toast"] button {
  color: var(--ink) !important;
  background: transparent !important;
}

footer { display: none !important; }
"""


In [ ]:
# =========================
# Gradio App
# =========================

with gr.Blocks(theme=gr.themes.Base(), css=CUSTOM_CSS, title="Course Desk｜課程公告轉待辦＋番茄鐘") as demo:
    announcement_state = gr.State(None)
    forum_items_state = gr.State([])
    detected_state = gr.State([])
    courses_state = gr.State([])
    forums_state = gr.State([])

    timer_remaining = gr.State(25 * 60)
    timer_running = gr.State(False)
    timer_phase = gr.State("work")
    timer_started_at = gr.State("")
    timer_task_id = gr.State("")
    timer_task_title = gr.State("")
    timer_count = gr.State(0)
    timer_work_total = gr.State(25)
    timer_break_total = gr.State(5)
    timer_clock = gr.Timer(value=1, active=False)

    with gr.Column(elem_id="desk-shell"):
        gr.HTML("""
        <div class='hero-wrap'>
          <div class='eyebrow'>Course Desk · Moodle Announcement to Tasks</div>
          <h1 class='hero-title'>把課程公告，整理成<br>可以執行的待辦。</h1>
          <div class='hero-subtitle'>
            先抓課程或貼公告網址，載入單則公告後交由 Gemini 抽取待辦；
            任務可勾選寫入 Google Sheet，也能手動新增並用番茄鐘追蹤專注紀錄。
          </div>
        </div>
        """)

        with gr.Row(equal_height=True):
            with gr.Column(scale=5):
                with gr.Group(elem_classes=["panel"]):
                    gr.HTML("""
                    <div class='section-title'><h3>課程與公告</h3></div>
                    <div class='section-note'>可先抓取 Moodle 課程清單，再選課程公告；也可以直接貼上單則公告或 forum 列表網址。</div>
                    """)
                    moodle_base = gr.Textbox(label="Moodle base URL", value=DEFAULT_MOODLE_BASE)
                    fetch_courses_btn = gr.Button("抓取我的課程", elem_classes=["secondary-btn"])
                    course_dropdown = gr.Dropdown(label="選擇課程", choices=[], value=None)
                    fetch_forums_btn = gr.Button("抓取此課程的公告入口", elem_classes=["secondary-btn"])
                    forum_dropdown = gr.Dropdown(label="選擇公告入口 / 討論區", choices=[], value=None)

                    url_input = gr.Textbox(label="公告網址", placeholder="可貼 /mod/forum/view.php 或 /mod/forum/discuss.php")
                    with gr.Row():
                        fetch_url_btn = gr.Button("抓取公告網址", elem_classes=["primary-btn"])
                        load_selected_btn = gr.Button("載入選取公告", elem_classes=["secondary-btn"])
                    announcement_dropdown = gr.Dropdown(label="若是公告列表，請選擇單則公告", choices=[], value=None)
                    announcement_html = gr.HTML("<div class='empty-note'>尚未載入公告。</div>")
                    import_status = gr.HTML(status_html("等待操作。"))
                    extract_btn = gr.Button("Gemini 解析目前公告", elem_classes=["primary-btn"])

                with gr.Group(elem_classes=["panel", "panel-green"]):
                    gr.HTML("""
                    <div class='section-title'><h3>行事曆同步</h3></div>
                    <div class='section-note'>如果 Moodle 有 calendar export，可貼上 .ics 網址，將未來截止事件同步成任務。</div>
                    """)
                    ics_input = gr.Textbox(label="Moodle Calendar ICS URL", placeholder="可留空，使用 Colab Secret: MOODLE_ICS_URL")
                    sync_ics_btn = gr.Button("同步 Moodle 行事曆", elem_classes=["secondary-btn"])

            with gr.Column(scale=7):
                with gr.Group(elem_classes=["panel", "panel-warm"]):
                    gr.HTML("""
                    <div class='section-title'><h3>待確認任務</h3></div>
                    <div class='section-note'>AI 解析後先從多選清單挑出要寫入的任務，不再整批無腦加入。</div>
                    """)
                    detected_table = gr.HTML(detected_tasks_table_html(None))
                    selected_tasks = gr.Dropdown(
                        label="選擇要寫入的任務（可多選）",
                        choices=[],
                        value=[],
                        multiselect=True,
                        interactive=True
                    )
                    save_tasks_btn = gr.Button("寫入選取任務", elem_classes=["primary-btn"])

                with gr.Group(elem_classes=["panel"]):
                    gr.HTML("""
                    <div class='section-title'><h3>手動新增待辦</h3></div>
                    <div class='section-note'>公告沒有寫清楚，或你想自己補任務時，直接從這裡新增。</div>
                    """)
                    with gr.Row():
                        manual_title = gr.Textbox(label="任務名稱", placeholder="例如：整理期末報告大綱")
                        manual_course = gr.Textbox(label="課程", placeholder="可留空")
                    with gr.Row():
                        manual_due = gr.Textbox(label="截止時間", placeholder="YYYY-MM-DD HH:MM，可留空")
                        manual_priority = gr.Dropdown(label="優先", choices=["high", "medium", "low"], value="medium")
                        manual_est = gr.Number(label="估時分鐘", value=25, precision=0, minimum=1, maximum=300)
                    manual_notes = gr.Textbox(label="備註", placeholder="可留空")
                    add_manual_btn = gr.Button("新增手動任務", elem_classes=["secondary-btn"])

        with gr.Row(equal_height=True):
            with gr.Column(scale=7):
                with gr.Group(elem_classes=["panel", "panel-blue"]):
                    gr.HTML("""
                    <div class='section-title'><h3>本週任務</h3></div>
                    <div class='section-note'>從多選清單選任務後可標記完成；同一份清單也會提供給番茄鐘選擇。</div>
                    """)
                    with gr.Row():
                        refresh_btn = gr.Button("重新整理任務", elem_classes=["secondary-btn"])
                        mark_done_btn = gr.Button("標記勾選任務完成", elem_classes=["secondary-btn"])
                    done_task_select = gr.Dropdown(
                        label="選擇要標記完成的任務（可多選）",
                        choices=[],
                        value=[],
                        multiselect=True,
                        interactive=True
                    )
                    tasks_table = gr.HTML(tasks_preview_table_html())

            with gr.Column(scale=5):
                with gr.Group(elem_classes=["panel"]):
                    gr.HTML("""
                    <div class='section-title'><h3>番茄鐘</h3></div>
                    <div class='section-note'>支援開始、暫停、結束；專注完成會寫入 Google Sheet，休息會自動切換。</div>
                    """)
                    timer_html = gr.HTML(timer_display_html())
                    task_dropdown = gr.Dropdown(label="選擇任務", choices=task_choice_pairs(), value=None)
                    with gr.Row():
                        work_min = gr.Number(label="專注分鐘", value=25, precision=0, minimum=1, maximum=180)
                        break_min = gr.Number(label="休息分鐘", value=5, precision=0, minimum=0, maximum=60)
                    timer_note = gr.Textbox(label="紀錄備註", placeholder="可留空")
                    with gr.Row():
                        start_timer_btn = gr.Button("開始 / 繼續", elem_classes=["primary-btn"])
                        pause_timer_btn = gr.Button("暫停", elem_classes=["secondary-btn"])
                        end_timer_btn = gr.Button("結束並紀錄", elem_classes=["secondary-btn"])
                    timer_status = gr.HTML(status_html("尚未開始。"))

        app_status = gr.HTML(status_html("系統就緒。"))

    fetch_courses_btn.click(
        fn=ui_fetch_courses,
        inputs=[moodle_base],
        outputs=[courses_state, course_dropdown, import_status]
    )

    fetch_forums_btn.click(
        fn=ui_fetch_course_forums,
        inputs=[course_dropdown],
        outputs=[forums_state, forum_dropdown, import_status]
    )

    forum_dropdown.change(
        fn=lambda x: x or "",
        inputs=[forum_dropdown],
        outputs=[url_input]
    )

    fetch_url_btn.click(
        fn=ui_fetch_announcement_url,
        inputs=[url_input],
        outputs=[announcement_state, forum_items_state, announcement_dropdown, announcement_html, import_status]
    )

    load_selected_btn.click(
        fn=ui_load_selected_announcement,
        inputs=[announcement_dropdown],
        outputs=[announcement_state, announcement_html, import_status]
    )

    extract_btn.click(
        fn=ui_extract_tasks,
        inputs=[announcement_state],
        outputs=[detected_state, detected_table, selected_tasks, import_status]
    )

    save_tasks_btn.click(
        fn=ui_save_selected_tasks,
        inputs=[selected_tasks, detected_state, announcement_state],
        outputs=[tasks_table, done_task_select, task_dropdown, app_status]
    )

    add_manual_btn.click(
        fn=ui_add_manual_task,
        inputs=[manual_title, manual_course, manual_due, manual_priority, manual_est, manual_notes],
        outputs=[manual_title, manual_course, manual_due, manual_priority, manual_est, manual_notes, tasks_table, done_task_select, task_dropdown, app_status]
    )

    refresh_btn.click(
        fn=ui_refresh_tasks,
        inputs=[],
        outputs=[tasks_table, done_task_select, task_dropdown, app_status]
    )

    mark_done_btn.click(
        fn=ui_mark_tasks_done,
        inputs=[done_task_select],
        outputs=[tasks_table, done_task_select, task_dropdown, app_status]
    )

    sync_ics_btn.click(
        fn=ui_sync_ics,
        inputs=[ics_input],
        outputs=[tasks_table, done_task_select, task_dropdown, app_status]
    )

    start_timer_btn.click(
        fn=start_or_resume_timer,
        inputs=[task_dropdown, work_min, break_min, timer_remaining, timer_running, timer_phase, timer_started_at, timer_task_id, timer_task_title, timer_count, timer_work_total, timer_break_total],
        outputs=[timer_remaining, timer_running, timer_phase, timer_started_at, timer_task_id, timer_task_title, timer_count, timer_work_total, timer_break_total, timer_html, timer_status, timer_clock],
        show_progress="hidden"
    )

    pause_timer_btn.click(
        fn=pause_timer,
        inputs=[timer_remaining, timer_running, timer_phase, timer_count],
        outputs=[timer_running, timer_html, timer_status, timer_clock],
        show_progress="hidden"
    )

    end_timer_btn.click(
        fn=finish_current_timer,
        inputs=[timer_remaining, timer_running, timer_phase, timer_started_at, timer_task_id, timer_task_title, timer_count, timer_work_total, timer_break_total, timer_note],
        outputs=[timer_remaining, timer_running, timer_phase, timer_started_at, timer_task_id, timer_task_title, timer_count, timer_html, timer_status, timer_clock],
        show_progress="hidden"
    )

    timer_clock.tick(
        fn=timer_tick,
        inputs=[timer_remaining, timer_running, timer_phase, timer_started_at, timer_task_id, timer_task_title, timer_count, timer_work_total, timer_break_total, timer_note],
        outputs=[timer_remaining, timer_running, timer_phase, timer_started_at, timer_task_id, timer_task_title, timer_count, timer_html, timer_clock],
        show_progress="hidden"
    )

# 在 Colab 中執行：若需要公開連結可改成 share=True
demo.launch(debug=True)


/tmp/ipykernel_1831/3954432475.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), css=CUSTOM_CSS, title="Course Desk｜課程公告轉待辦＋番茄鐘") as demo:
/tmp/ipykernel_1831/3954432475.py:5: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), css=CUSTOM_CSS, title="Course Desk｜課程公告轉待辦＋番茄鐘") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3fda6639ad41dfd723.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2411.91ms
